# Synthesis: decision features, frozen embeddings, supervised TAR and native text systems

**Why this notebook.** It brings Experiments 1–4 together: within each fixed predictor, are <span style="color:#5A6E8C">decision features</span> better than <span style="color:#987448">frozen embeddings</span>, and how do these and supervised TAR compare with <span style="color:#4F8075">native text systems</span> that change the predictor as well? Headline metrics are fixed before results; no best recipe or model is selected here.

| Figure | Question | Shows |
|---|---|---|
| Decision versus embedding (<span style="color:#6A51A3">binary</span>, <span style="color:#008B8B">multiclass</span>, <span style="color:#B8860B">regression</span>) | Do decision features beat general embeddings? | paired difference per extractor × mode × predictor, equal-weight average across complete dataset × encoder/PCA pairs, with detailed recipes retained in tables |
| Decision versus TAR (table) | How close do decision features come to supervised TAR? | paired difference between each decision recipe and both TAR PCA recipes within the same predictor, dataset and folds |
| Versus native systems | How far are the pipelines from native text systems? | every fixed-predictor recipe minus each native system on matching folds |

## 1. Scope and provenance

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.visualize import figures, style
style.apply()
NOTEBOOK = '8. synthesis/01_representation_comparison'
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and saved evidence")
display_json("Notebook execution identity", provenance())
from src.evaluation.results import collect
from src.evaluation.notebook_study import planned_coverage, headline_summary, burden_table, contrast_summary
from src.visualize.study import coverage_figures, native_figures, burden_figures
from src.evaluation.endpoints import HEADLINES
status, folds, per_dataset, paired = collect()
summary = headline_summary(folds)
display_json("Saved result coverage", status)
display_json("Prespecified headline metrics", HEADLINES)
print("Only saved, validated folds are read. No fit, API call, download or hypothesis test runs here.")


Only saved, validated folds are read. No fit, API call, download or hypothesis test runs here.


## 2. The complete study grid and coverage

These are planned configurations, not independent statistical observations. Binary inner threshold fits are counted separately.

In [2]:
report_section('2. The complete study grid and coverage')
from src.evaluation.campaign import benchmark_plan
from src.evaluation.notebook_evidence import study_grid
grid = study_grid(benchmark_plan())
display_table("Main grid, outer fits and binary inner fits", grid, preview_rows=3)
display_table("Complete primary headline results and exact recipe identities", summary[summary.phase.isin(["experiment_1", "experiment_2", "experiment_3", "experiment_4"])])

experiment,datasets,predictors,configurations,outer_fits,binary_inner_fits
experiment_1,18,6,3996,19980,9990
experiment_2,18,6,648,3240,1620
experiment_3,18,6,216,1080,0
experiment,datasets,predictors,configurations,outer_fits,binary_inner_fits
experiment_1,18,6,3996,19980,9990
experiment_2,18,6,648,3240,1620
experiment_3,18,6,216,1080,0
experiment_4,18,3,54,270,135


arm_id,phase,dataset_id,task_type,model_id,checkpoint_id,feature_set,provider,encoder,pca_scope,pca_components,split_id,metric,mean,descriptive_fold_sd,defined_folds


## 3. Decision features versus every frozen and supervised embedding recipe

Match dataset, predictor, checkpoint, split and fold. Each cell averages complete dataset effects, with coverage retained. J64 means 64 joint components and P32 means 32 components per text column.

In [3]:
report_section('3. Decision features versus every embedding recipe')
from src.evaluation.representation_views import decision_vs_embedding, measured_extraction
from src.visualize.representation_views import direct_comparison_figures
contrasts = decision_vs_embedding(folds)
display_table("Direct matched recipe effects and contributing dataset identities", contrast_summary(contrasts))
for fig, name, caption in direct_comparison_figures(contrasts):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


tar_direct = decision_vs_embedding(folds, phase="experiment_3")
display_table("Decision features minus supervised TAR", contrast_summary(tar_direct))
for fig, name, caption in direct_comparison_figures(tar_direct, family="supervised TAR"):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)

### 3b. Target-aware versus frozen E5

The same encoder and PCA budget are paired within the same predictor and folds. The contrast includes the documented upstream input and scaling choices, not only label-based adaptation.

In [4]:
report_section("3b. Matched frozen-E5 comparison")
from src.evaluation.representation_views import tar_vs_frozen
tar_comparison = tar_vs_frozen(folds)
display_table("TAR minus frozen E5-small with the same predictor/PCA/split", contrast_summary(tar_comparison))
print("This compares complete pipelines: prefix, missing placeholders, token coverage and coordinate standardization also differ. AUC and R² do not depend on the binary threshold policy.")


This compares complete pipelines: prefix, missing placeholders, token coverage and coordinate standardization also differ. AUC and R² do not depend on the binary threshold policy.


## 4. Where do native-text systems sit?

This explicitly changes the prediction engine. Every fixed recipe is compared with every native reference on matching folds. No family is represented only by its test-selected winner.

In [5]:
report_section('4. Where do native-text systems sit?')
from src.evaluation.notebook_study import cross_system_contrasts
from src.visualize.study import cross_system_figures
system_contrasts = cross_system_contrasts(folds)
display_table("Descriptive cross-system differences and matched coverage", contrast_summary(system_contrasts))
for fig, name, caption in cross_system_figures(system_contrasts):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 5. Observed extraction cost, fit time and representation width

Full extraction timings are distinct from pilots, retries and cache reuse. Hardware/run identities must accompany any cost-performance statement.

In [6]:
report_section('5. Observed extraction cost, fit time and representation width')
display_table("Measured full-extraction timing summaries", measured_extraction())
display_table("Saved predictive compute/width summaries", burden_table(folds, ["experiment_1", "experiment_2", "experiment_3", "experiment_4"]))
record("Outcomes describe the selected 18 public tasks under shuffled row CV. Pretraining overlap can be unknown; repeated-text cohorts and absence of row-label inputs do not prove contamination-free pretrained weights. No raw target-unit errors or AUC/R-squared scores are pooled across task types.", "Scope of conclusions")

run_id,extractor,context,stage,dataset_id,observations,items,timed_observations,wall_seconds,mean_seconds_per_observation,min_seconds,max_seconds,failures,cuda_synchronized_observations,source
25db27bed5fe402aa7ccb61a1030e3fc,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769380"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",decision_inference,REG_TEXT_SCIMAGOJR_IMPACT,69643,139285,69643,4640.263308,0.066629,0.057781,15.909910,0,69643,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_25db27bed5fe402aa7ccb61a1030e3fc.json
25db27bed5fe402aa7ccb61a1030e3fc,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769380"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",model_loading,None,1,0,1,7.570737,7.570737,7.570737,7.570737,0,1,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_25db27bed5fe402aa7ccb61a1030e3fc.json
3a2bbcb3b8de4b1fa94d0fc16f7e4440,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769381"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",decision_inference,REG_TEXT_VANCOUVER_SALARIES,14770,29540,14770,896.404224,0.060691,0.057792,16.008456,0,14770,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_3a2bbcb3b8de4b1fa94d0fc16f7e4440.json
3a2bbcb3b8de4b1fa94d0fc16f7e4440,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769381"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",model_loading,None,1,0,1,26.515223,26.515223,26.515223,26.515223,0,1,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_3a2bbcb3b8de4b1fa94d0fc16f7e4440.json
43d4e95a20294409a61bf6ad90d869a9,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769372"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",decision_inference,MUL_TEXT_WINE_REVIEW,79160,158319,79160,11556.755709,0.145992,0.103110,16.034905,0,79160,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_43d4e95a20294409a61bf6ad90d869a9.json
43d4e95a20294409a61bf6ad90d869a9,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769372"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",model_loading,None,1,0,1,11.981525,11.981525,11.981525,11.981525,0,1,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_43d4e95a20294409a61bf6ad90d869a9.json
4784346df78d4f5696ae0003e25002ce,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769365"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",decision_inference,BIN_TEXT_FAKE_JOB_POSTING,17316,34631,17316,1128.458924,0.065169,0.039403,15.979249,0,17316,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_4784346df78d4f5696ae0003e25002ce.json
4784346df78d4f5696ae0003e25002ce,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769365"", ""settings_sha256"": ""a255ce6d98b8c0c095bf516028082c5da53cd6c9f9098863df83107946e36bd3""}",model_loading,None,1,0,1,36.621262,36.621262,36.621262,36.621262,0,1,C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\2. feature_creation\manifests\timings\cloudflare_4784346df78d4f5696ae0003e25002ce.json
run_id,extractor,context,stage,dataset_id,observations,items,timed_observations,wall_seconds,mean_seconds_per_observation,min_seconds,max_seconds,failures,cuda_synchronized_observations,source
25db27bed5fe402aa7ccb61a1030e3fc,cloudflare,"{""cluster"": ""mindwell"", ""job_id"": ""11769380"", 

## Complete printed report

Everything displayed above is printed in section order. Individual predictions and the full metric store remain available in the referenced evaluation databases.

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nNo fitting, inference or outcome-based method selection was performed.")

8. synthesis/01_representation_comparison

1. Scope, provenance and saved evidence


Notebook execution identity
{
  "code_sha256": "42e503ac9eb6eb9df4b98562e672443feb73923ed9f8c16fa4358f861569d5db",
  "git_head": "2e1e0a5d193f108c7cc59c19820690023aaacecb",
  "git_dirty": true,
  "tfm_library_pin": "286157a5f8a4b6c53cf1341309e81bd1ff4ce128 dm-library (heads/main)\n 8f14e5945ca6b068ed0c76218e769d3e672d0d13 tfm-library (heads/main)\n 914ef4e57e1f153080e2686d03179c223db38c9a vsc-library (914ef4e)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Saved result coverage
{
  "status": "no_completed_folds"
}


Prespecified headline metrics
{
  "binary_classification": "roc_auc",
  "multiclass_classification": "roc_auc_ovr_macro",
  "regression": "r2"
}


2. The complete study grid and coverage


Main grid, outer fits and binary inner fits
  experiment  datasets  predictor